<a href="https://colab.research.google.com/github/hsliz31/mongodb/blob/main/VectorSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Embedding Vector

"강아지" = [0.245, 0.764, -0.314, 0.456, -0.342, ...]

"고양이" = [0.252, 0.777, -0.314, 0.456, -0.342, ...]


- 각 차원은 데이터의 특성을 나타냅니다.


의미가 유사한 임베딩 벡터는
의미 공간에서 가까이 위치합니다.

### Multimodal Embedding Model

다양한 형태의 데이터를 공통된 의미공간에 표현하기도 합니다.


- 소스 데이터
  - text
  - code
  - image
  - sound
  - video




**Atlas Vector Search** :
문서와 함께 벡터를 저장합니다



표지데이터를 바탕으로 책 문서

## Step 1: Setup prerequisite

## Step 2: Import data into MongoDB

## Step 3: Generating embeddings

`len(embedding.embeddings[0]`


차원수를 확인 / Feature #

`def get_embeddings`

- 이미지면 다운로드 받아서 인코딩
- 텍스트는 그대로 인코딩

`collection`

- 문서 집합
- 커서를 받아옴

`field_to_embed`
- 제목페이지

$set

- 존재 하면 덮어씌우기
- 존재 하지 않으면 추가하는 업데이트 명세

순회하고 있는 call


## Step 4. Adding embeddings to existing data in Atlas

## Step 5. Create a vector search index

벡터 검색 쿼리
- 쿼리를 동일한 임베딩 모델로 인코딩하여 검색을 수행합니다.


Hierarchical Naviagble Small Worlds (HNSW)

MongoDB Vector Search 의 구조

- 벡터를 노드로, 벡터 공간에서 거리를 기반으로 엣지를 생성하는 계층적 연결 그래프 구조
- ANN 알고리즘으로, 각 계층의 엔트리포인트에서 시작하여 상위 계층 (희소 연결) 에서 하위 계층 (조밀 연결)으로 점진적 탐색
- 대규모 데이터셋에서 효율적인 검색 방법 제공

### 벡터 공간에서 유사도를 측정하는 방법

3가지 유사도 검색 방법

1. **Cosine Similarity**
- 벡터 간 각도를 측정
- 방향적 유사성을 나타냄
- 크기에 영향을 받지 않음

2. Euclidean
3. Dot product (내적)


3 Steps
1. 문서 내 임베딩 생성
2. 벡터 검색 인덱스 생성
3. 벡터 검색 쿼리

#### 벡터 검색 인덱스 생성



In [ ]:
model = {
    "name": ATLAS_VECTOR_SEARCH_INDEX_NAME,
    "type": "vectorSearch",
    "definition": {
        "fields": [
            {
                "type": "vector",
                "path": "embedding",
                "numDimensions": 1024,
                "similarity": "cosine",
            }
        ]
    },
}

벡터 검색 쿼리 수행
- 몽고DB

- `limit` : 쿼리벡터와 가장 유사한 n 개수 가져옴 (몇개 가져올꺼야?)
- `numCandidates` :
  - ANN 수행할때 쓰이는 값 ->
  - 시스템이 고려할 때 후보개수
    - 너무 많으면 충분히 ANN 강점을 못누림
    - 너무 적으면 의미가 없음
    - 정확도 / 속도 tradeoff 함
    - guide: limit 의 10~20배

## Step 6. Perform vector search queries

In [ ]:
# Define a function to retrieve relevant documents for a user query using vector search
def vector_search(
    user_query: str, mode: str, filter: Optional[Dict] = {}
) -> None:
    """
    Retrieve relevant documents for a user query using vector search.

    Args:
    user_query (str): The user's query (can be a piece of text or a link to an image)
    mode (str): Query mode (image or text)
    filter (Optional[Dict], optional): Optional vector search pre-filter
    """
    # Generate embedding for the `user_query` using the `get_embeddings` function defined in Step 4
    # `input_type` should be set to "query" since we are embedding the query
    query_embedding = get_embeddings(user_query, mode, "query")

    # Define an aggregation pipeline consisting of a $vectorSearch stage, followed by a $project stage
    # Set the number of candidates to 20 and only return the top 5 documents from the vector search
    # Set the `filter` field in the $vectorSearch stage to the value `filter` passed to the function
    # In the $project stage, exclude the `_id` field, include these fields: `title`, `cover`, `year`, `pages`, and the `vectorSearchScore`
    # NOTE: Use variables defined previously for the `index`, `queryVector` and `path` fields in the $vectorSearch stage
    pipeline = [
    {
        "$vectorSearch": {
            "index": ATLAS_VECTOR_SEARCH_INDEX_NAME,
            "queryVector": query_embedding,
            "path": "embedding",
            "numCandidates": 20,
            "filter": filter,
            "limit": 5,
        }
    },
    {"$project": {"_id": 0, "title": 1, "cover": 1, "year":1, "pages":1, "score": {"$meta": "vectorSearchScore"}}},
    #비주얼로 보여주는 개념
  ]

    # Execute the aggregation `pipeline` and store the results in `results`
    results = collection.aggregate(pipeline)

    # Print book title, score, and cover image
    for book in results:
        cover = Image.open(requests.get(book.get("cover"), stream=True).raw).resize((100,150))
        print(f"{book.get('title')}({book.get('year')}, {book.get('pages')} pages): {book.get('score')}")
        display(cover)

aggregation pipeline

In [ ]:
# Test the vector search with a text query
vector_search("animal", "text")

# Also try these text queries:
# - A rainbow of lively colors
# - Creatures wondrous or familiar
# - A boy and the ocean
# - Houses

이미지로 이미지를 검색

In [ ]:
# Test the vector search with an image query
vector_search("https://images.isbndb.com/covers/10835953482746.jpg", "image")

# Also try these image queries:
# - ../data/images/salad.jpg
# - ../data/images/kitten.png
# - ../data/images/barn.png

## Step 7. Adding pre-filters to your vector search

vector search (비쌈)


**튜닝 방법 2가지**

1. 사전 필터링 (pre-filtering)
- 필터링을 먼저 했을 때 이점
- 새벽배송,
- 정확도 (limit)
2. 벡터 양자화 (Quantization)
- 원본 정밀도의 벡터를 더 적은 비트로 추속하는 과정
  - scalar
    - 해상도는 낮아지겠지만 비트 수는 줄일수 있음
  - binary
    - 각 벡터의 차원값을 특정 기준점을 기준으로 0 또는 1로만 표현함
    

50개 책, 512 차원, 4바이트
